DEEP NEURAL NETWORKS - ASSIGNMENT 3: RNN vs TRANSFORMER FOR TIME SERIES

Recurrent Neural Networks vs Transformers for Time Series Prediction

STUDENT INFORMATION

Name: Ajay Kumar Pandit

ASSIGNMENT OVERVIEW

This assignment requires you to implement and compare two models for time
series prediction:
1. Recurrent Neural Network (RNN/LSTM) architecture
2. Transformer-based architecture

Learning Objectives:
- Build sequence models for forecasting
- Compare recurrent and attention-based approaches
- Evaluate forecasting quality with standard metrics
- Use industry-standard deep learning frameworks


In [2]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import LSTM, Dense, Input, MultiHeadAttention, LayerNormalization
import time, json

# Install yfinance if needed
try:
    import yfinance as yf
except:
    !pip install yfinance
    import yfinance as yf


In [3]:

df = yf.download("AAPL", start="2015-01-01", end="2024-01-01")
data = df[['Close']].values

dataset_name = "Apple Stock Prices"
dataset_source = "Yahoo Finance (yfinance)"
n_samples = len(data)
n_features = 1
sequence_length = 20
prediction_horizon = 1
problem_type = "time_series_forecasting"

primary_metric = "RMSE"
metric_justification = "RMSE penalizes large forecasting errors."

print("Samples:", n_samples)


[*********************100%***********************]  1 of 1 completed

Samples: 2264


In [4]:

scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(data)

def create_sequences(data, seq_len):
    X, y = [], []
    for i in range(len(data)-seq_len):
        X.append(data[i:i+seq_len])
        y.append(data[i+seq_len])
    return np.array(X), np.array(y)

X, y = create_sequences(data_scaled, sequence_length)

split = int(len(X)*0.9)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]


In [5]:

model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(sequence_length,1)),
    LSTM(32),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')

start = time.time()
history = model.fit(X_train, y_train, epochs=15, batch_size=32, verbose=1)
rnn_time = time.time() - start

rnn_initial_loss = history.history['loss'][0]
rnn_final_loss = history.history['loss'][-1]


Epoch 1/15


/Users/ajay/Library/Python/3.9/lib/python/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0321
Epoch 2/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 5.7476e-04
Epoch 3/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 5.3216e-04
Epoch 4/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 5.2147e-04
Epoch 5/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 4.5874e-04
Epoch 6/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 4.5307e-04
Epoch 7/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 5.9120e-04
Epoch 8/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 4.4245e-04
Epoch 9/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 3.9348e-04
Epoch 10/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 4.6709e-04
Epoch 11/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 3.9504e-04
Epoch 12/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 3.6259e-04
Epoch 13/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 3.9001e-04
Epoch 14/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 3.5504e-04
Epoch 15/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 3m

In [6]:

y_pred = model.predict(X_test)

y_test_inv = scaler.inverse_transform(y_test.reshape(-1,1))
y_pred_inv = scaler.inverse_transform(y_pred)

rnn_mae = mean_absolute_error(y_test_inv, y_pred_inv)
rnn_rmse = np.sqrt(mean_squared_error(y_test_inv, y_pred_inv))
rnn_mape = np.mean(np.abs((y_test_inv - y_pred_inv)/y_test_inv))*100
rnn_r2 = r2_score(y_test_inv, y_pred_inv)

print(rnn_mae, rnn_rmse, rnn_mape, rnn_r2)


8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
3.5121794976128466 4.2628153113885 2.0004430678847798 0.901459636091268


In [7]:

def positional_encoding(seq_len, d_model):
    pos = np.arange(seq_len)[:, None]
    i = np.arange(d_model)[None, :]
    angle = pos / np.power(10000, (2*(i//2))/d_model)
    pe = np.zeros((seq_len, d_model))
    pe[:, 0::2] = np.sin(angle[:, 0::2])
    pe[:, 1::2] = np.cos(angle[:, 1::2])
    return pe


In [8]:

inputs = Input(shape=(sequence_length,1))
x = Dense(64)(inputs)

x = x + positional_encoding(sequence_length, 64)

attn = MultiHeadAttention(num_heads=4, key_dim=64)(x,x)
x = LayerNormalization()(x + attn)

x = Dense(32, activation='relu')(x)
outputs = Dense(1)(x)

transformer = Model(inputs, outputs)
transformer.compile(optimizer='adam', loss='mse')

start = time.time()
history_t = transformer.fit(X_train, y_train, epochs=15, batch_size=32)
transformer_time = time.time() - start

transformer_initial_loss = history_t.history['loss'][0]
transformer_final_loss = history_t.history['loss'][-1]


Epoch 1/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.3966
Epoch 2/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0030
Epoch 3/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0020
Epoch 4/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0016
Epoch 5/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0013
Epoch 6/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0010
Epoch 7/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 7.6731e-04
Epoch 8/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 6.0792e-04
Epoch 9/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 5.6201e-04
Epoch 10/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 3.5927e-04
Epoch 11/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 3.8501e-04
Epoch 12/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 3.8312e-04
Epoch 13/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 6.8161e-04
Epoch 14/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 3.0190e-04
Epoch 15/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - 

In [9]:

y_pred_t = transformer.predict(X_test)
y_pred_t = y_pred_t[:, -1, :]

y_pred_t_inv = scaler.inverse_transform(y_pred_t)

transformer_mae = mean_absolute_error(y_test_inv, y_pred_t_inv)
transformer_rmse = np.sqrt(mean_squared_error(y_test_inv, y_pred_t_inv))
transformer_mape = np.mean(np.abs((y_test_inv - y_pred_t_inv)/y_test_inv))*100
transformer_r2 = r2_score(y_test_inv, y_pred_t_inv)

print(transformer_mae, transformer_rmse, transformer_mape, transformer_r2)


8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step
5.714551595052083 6.247035901228619 3.245946978235642 0.7883739473490395


In [10]:

results = {
    "dataset_name": dataset_name,
    "n_samples": int(n_samples),
    "rnn_loss_decreased": rnn_final_loss < rnn_initial_loss,
    "transformer_loss_decreased": transformer_final_loss < transformer_initial_loss
}
print(json.dumps(results, indent=2))


{
  "dataset_name": "Apple Stock Prices",
  "n_samples": 2264,
  "rnn_loss_decreased": true,
  "transformer_loss_decreased": true
}
